# Export for the web app
Writes the JSON files the Next.js app reads, exactly as defined in `notebooks/data-contract.md`
(schema version 1). The contract is the spec; this notebook follows it section by section.

- Layouts for Q1, Q2 and Q3 are **re-fitted on all five years** with `stations.py`.
- Every stored metric is computed from the **exported** values (coordinates rounded to 5 decimals,
  integer weights), so the browser can reproduce it.
- `evidence.json` is copied from notebook 05's saved results, not recomputed.
- All contract checks run first. If one fails, nothing is written.

## Config

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display

folder = Path.cwd().resolve()
REPO_ROOT = next((p for p in (folder, *folder.parents) if (p / "CLAUDE.md").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Launch from inside the repository.")
sys.path.insert(0, str(REPO_ROOT / "notebooks"))
import stations as st

SCHEMA_VERSION = 1
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
RESULTS_DIR = PROCESSED_DIR / "results"          # notebook 05; used for evidence.json only
OUTPUT_DIR = REPO_ROOT / "web" / "public" / "data"

REGIONS = ["Cariboo", "Coastal", "Kamloops", "Northwest", "Prince George", "Southeast"]
EXPECTED_FIRES, EXPECTED_CANDIDATES, EXPECTED_HALLS, EXPECTED_STATIC = 7032, 851, 521, 195
Q1_K_MAX = 60
Q2_K_MAX = 200
TARGET_RELATIVE = 0.95
ELBOW_RANGE = [1, 60]
NEXT_N = 20
HALL_BUDGET_FACTOR = 2
CONFORMANCE_K = 20
SHARE_TOLERANCE, MINUTES_TOLERANCE = 0.0001, 0.1
MAX_OTHER_FILES_BYTES, MAX_FIRES_BYTES = 1_000_000, 1_500_000
FIRMS_PRODUCT = "VIIRS_SNPP_SP (2022-07-27..2022-08-10 filled from VIIRS_NOAA20_SP)"
Q1_LABELS = {"capped_pmedian": "Fastest response (capped p-median)",
             "fair": "One station per fire centre first (fair)"}

model = st.load_model()
config = model["config"]
results = json.loads((RESULTS_DIR / "summary.json").read_text(encoding="utf-8"))
ignitions = pd.read_parquet(PROCESSED_DIR / "ignitions.parquet")
ignitions_meta = json.loads((PROCESSED_DIR / "ignitions_meta.json").read_text(encoding="utf-8"))

THRESHOLD_MIN = config["THRESHOLD_MIN"]
CAP_MIN = results["CAP_MIN"]
TRUCK_PERCENTILE = results["TRUCK_PERCENTILE"]
TRUCK_BUSY_DAYS_ONLY = results["TRUCK_BUSY_DAYS_ONLY"]
GENERATED_AT = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def share(value):
    return round(float(value), 4)


def minutes(value):
    return round(float(value), 1)


def header():
    return {"schema_version": SCHEMA_VERSION, "generated_at": GENERATED_AT}

## 1. All-years demand
Stack train on top of test: same candidate columns, rows in the same order as the demand table.

In [2]:
candidates = model["candidates"]
demand = pd.concat([model["demand_train"], model["demand_test"]], ignore_index=True)
cost_fit = np.vstack([model["cost_train"], model["cost_test"]])   # float32, used by the solvers
assert cost_fit.shape == (EXPECTED_FIRES, EXPECTED_CANDIDATES), cost_fit.shape
assert len(demand) == EXPECTED_FIRES and len(candidates) == EXPECTED_CANDIDATES
assert sorted(set(demand["region"]) | set(candidates["region"])) == REGIONS
assert (demand["weight"] % 1 == 0).all()
halls = np.flatnonzero((candidates["kind"] == "hall").to_numpy())
assert halls.tolist() == list(range(EXPECTED_HALLS))
weights = demand["weight"].to_numpy().astype(int)
regions_demand = demand["region"].to_numpy()
regions_cand = candidates["region"].to_numpy()
print(f"{len(demand):,} fires x {len(candidates)} candidates; years {sorted(demand['year'].unique().tolist())}")

7,032 fires x 851 candidates; years [2019, 2020, 2021, 2022, 2023]


## 2. Fit on all five years
The solvers use the float32 matrices, as in notebook 05. Only the pick orders are kept from this step;
all metrics are computed in the next step.

In [3]:
capped_fit = np.minimum(cost_fit, CAP_MIN)
q1_picks = {"capped_pmedian": st.greedy_pmedian(capped_fit, weights, Q1_K_MAX)["picks"]}
fair_fit = st.greedy_fair(capped_fit, weights, regions_demand, regions_cand, Q1_K_MAX)
q1_picks["fair"] = fair_fit["picks"]
fair_seeds = fair_fit["seeds"]
q2_picks = st.greedy_coverage(cost_fit, weights, Q2_K_MAX, THRESHOLD_MIN)["picks"]
next_picks = st.greedy_coverage(cost_fit, weights, NEXT_N, THRESHOLD_MIN, fixed=halls)["picks"]
print(f"Q1 capped p-median: {len(q1_picks['capped_pmedian'])} picks; fair: {len(q1_picks['fair'])} picks, seeds {fair_seeds}")
print(f"Q2 greedy coverage: {len(q2_picks)} picks; next stations after the halls: {len(next_picks)}")

Q1 capped p-median: 60 picks; fair: 60 picks, seeds [772, 59, 430, 503, 505, 292]
Q2 greedy coverage: 200 picks; next stations after the halls: 20


## 3. Scoring matrix from the exported values
Contract section 2: round coordinates to 5 decimals, use integer weights, and recompute travel minutes
in float64 with the contract's formula. Every stored metric below comes from this matrix.

In [4]:
cand_lat, cand_lon = candidates["lat"].round(5).to_numpy(), candidates["lon"].round(5).to_numpy()
fire_lat, fire_lon = demand["lat"].round(5).to_numpy(), demand["lon"].round(5).to_numpy()


def contract_minutes(lat1, lon1, lat2, lon2):
    """minutes = haversine_km * detour / speed_kmh * 60 + dispatch_min (rows: fires, columns: sites)."""
    lat1, lon1 = np.radians(lat1)[:, None], np.radians(lon1)[:, None]
    lat2, lon2 = np.radians(lat2)[None, :], np.radians(lon2)[None, :]
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    km = 2 * config["EARTH_RADIUS_KM"] * np.arcsin(np.sqrt(a))
    return km * config["DETOUR"] / config["SPEED_KMH"] * 60 + config["DISPATCH_MIN"]


cost = contract_minutes(fire_lat, fire_lon, cand_lat, cand_lon)   # float64, used for every stored metric
assert cost.dtype == np.float64 and cost.shape == cost_fit.shape
print(f"Largest difference from the float32 fitting matrix: {np.abs(cost - cost_fit).max():.5f} min")

nearest = cost.min(axis=1)
reachable = nearest <= THRESHOLD_MIN          # decided before nearest_min is rounded
all_open = st.evaluate(cost, weights, np.arange(len(candidates)), THRESHOLD_MIN, regions=regions_demand)
CEILING = all_open["ceiling"]
CEILING_BY_REGION = all_open["by_region"]["ceiling"]
print(f"All-years ceiling: {CEILING:.4f}; unreachable fires: {(~reachable).sum():,}")
display(CEILING_BY_REGION.round(4).to_frame("ceiling"))


def score(sites):
    return st.evaluate(cost, weights, sites, THRESHOLD_MIN, regions=regions_demand)


def region_scores(by_region):
    return {region: {"coverage": share(by_region.at[region, "coverage"]),
                     "relative": share(by_region.at[region, "coverage_relative"])} for region in REGIONS}


def trucks_and_peaks(result, sites):
    """Trucks (busy-day percentile, minimum 1), peak simultaneous fires and covered-fire counts per site."""
    covered = result["minutes"] <= THRESHOLD_MIN
    served, assignment = demand[covered], result["assignment"][covered]
    trucks = st.trucks_per_station(served, assignment, percentile=TRUCK_PERCENTILE, busy_days_only=TRUCK_BUSY_DAYS_ONLY)
    peaks = st.trucks_per_station(served, assignment, percentile=100)
    counts = pd.Series(assignment).value_counts()
    sites = list(sites)
    return (trucks.reindex(sites, fill_value=1).astype(int), peaks.reindex(sites, fill_value=0).astype(int),
            counts.reindex(sites, fill_value=0).astype(int))

Largest difference from the float32 fitting matrix: 0.00156 min
All-years ceiling: 0.6818; unreachable fires: 1,935


,ceiling
region,
Cariboo,0.7358
Coastal,0.8676
Kamloops,0.9580
Northwest,0.6852
Prince George,0.4238
Southeast,0.8766


## 4. `meta.json`, `candidates.json`, `fires.json`, `static_fires.json`

In [5]:
files = {}

files["meta.json"] = {
    **header(),
    "data": {"firms_product": FIRMS_PRODUCT, "years": sorted(int(y) for y in demand["year"].unique()),
             "train_years": config["TRAIN_YEARS"], "test_year": config["TEST_YEARS"][0],
             "osm_query_date": config["osm_query_date"], "weather_used": False},
    "settings": {"threshold_min": THRESHOLD_MIN, "detour": config["DETOUR"], "speed_kmh": config["SPEED_KMH"],
                 "dispatch_min": config["DISPATCH_MIN"], "earth_radius_km": config["EARTH_RADIUS_KM"],
                 "reach_km": config["reach_km"], "cap_min": CAP_MIN, "truck_percentile": TRUCK_PERCENTILE,
                 "truck_busy_days_only": TRUCK_BUSY_DAYS_ONLY, "weight_cap": int(ignitions_meta["weight_cap"]),
                 "link_km": ignitions_meta["link_km"], "link_days": ignitions_meta["link_days"]},
    "regions": REGIONS,
    "counts": {"fires": len(demand),
               "fires_by_year": {str(year): int(count) for year, count in demand.groupby("year").size().items()},
               "static_fires_excluded": int(ignitions["static_suspect"].sum()), "candidates": len(candidates),
               "halls": int((candidates["kind"] == "hall").sum()), "towns": int((candidates["kind"] == "town").sum())},
    "ceiling": {"all_years": share(CEILING), "train": share(results["ceiling_train"]), "test": share(results["ceiling_test"]),
                "by_region": {region: share(CEILING_BY_REGION[region]) for region in REGIONS}},
}

files["candidates.json"] = {**header(), "candidates": [
    {"cand_id": int(row.cand_id), "name": None if pd.isna(row.name_) else str(row.name_), "kind": row.kind,
     "region": row.region, "lat": float(lat), "lon": float(lon),
     "population": None if pd.isna(row.population) else int(row.population)}
    for row, lat, lon in zip(candidates.rename(columns={"name": "name_"}).itertuples(index=False), cand_lat, cand_lon)]}

fire_rows = pd.DataFrame({
    "fire_id": demand["fire_id"].astype(str), "year": demand["year"].astype(int),
    "date": demand["first_date"].dt.strftime("%Y-%m-%d"), "lat": fire_lat, "lon": fire_lon,
    "weight": weights, "active_days": demand["active_days_capped"].astype(int), "region": regions_demand,
    "nearest_min": np.round(nearest, 1), "reachable": reachable,
}).sort_values(["date", "fire_id"])
files["fires.json"] = {**header(), "fires": [
    {"fire_id": r.fire_id, "year": int(r.year), "date": r.date, "lat": float(r.lat), "lon": float(r.lon),
     "weight": int(r.weight), "active_days": int(r.active_days), "region": r.region,
     "nearest_min": float(r.nearest_min), "reachable": bool(r.reachable)} for r in fire_rows.itertuples(index=False)]}

static = ignitions[ignitions["static_suspect"]].sort_values("fire_id")
files["static_fires.json"] = {**header(), "fires": [
    {"fire_id": str(r.fire_id), "year": int(r.year), "lat": round(float(r.lat), 5), "lon": round(float(r.lon), 5)}
    for r in static.itertuples(index=False)]}
print({name: len(next(v for k, v in payload.items() if isinstance(v, list))) if name != "meta.json" else "settings"
       for name, payload in files.items()})

{'meta.json': 'settings', 'candidates.json': 851, 'fires.json': 7032, 'static_fires.json': 195}


## 5. `q1_layouts.json`
Both variants to K = 60. The capped p-median curve has K = 1–60; the fair curve starts at K = 6
(55 entries), after its six region seeds.

In [6]:
def q1_curve(picks, first_k):
    curve = []
    for k in range(first_k, len(picks) + 1):
        result = score(picks[:k])
        curve.append({"k": k, "site": int(picks[k - 1]), "coverage": share(result["coverage"]),
                      "relative": share(result["coverage_relative"]), "mean_min": minutes(result["mean_minutes"]),
                      "by_region": region_scores(result["by_region"])})
    return curve


files["q1_layouts.json"] = {
    **header(), "fitted_on": files["meta.json"]["data"]["years"], "k_max": Q1_K_MAX, "default_variant": "capped_pmedian",
    "variants": {
        "capped_pmedian": {"label": Q1_LABELS["capped_pmedian"], "picks": [int(s) for s in q1_picks["capped_pmedian"]],
                           "curve": q1_curve(q1_picks["capped_pmedian"], 1)},
        "fair": {"label": Q1_LABELS["fair"], "seeds": [int(s) for s in fair_seeds],
                 "picks": [int(s) for s in q1_picks["fair"]], "curve": q1_curve(q1_picks["fair"], len(fair_seeds))},
    }}
q1_report = pd.DataFrame([{"variant": name, "k": entry["k"], "coverage": entry["coverage"], "relative": entry["relative"],
                           "mean_min": entry["mean_min"]}
                          for name, variant in files["q1_layouts.json"]["variants"].items()
                          for entry in variant["curve"] if entry["k"] in (10, 20, 40, 60)])
display(q1_report.set_index(["variant", "k"]))

coverage  relative  mean_min
variant        k                               
capped_pmedian 10    0.1874    0.2748     144.4
               20    0.3010    0.4415     114.3
               40    0.4706    0.6902      89.0
               60    0.5673    0.8321      76.2
fair           10    0.1727    0.2533     145.8
               20    0.2960    0.4342     114.5
               40    0.4631    0.6792      89.3
               60    0.5600    0.8214      76.4

## 6. `q2_coverage.json`
Greedy coverage to K = 200. `k_star` is the first K reaching 95% of the ceiling; `elbow_k` is the elbow
of the curve over K = 1–60. Trucks are sized on the fires each station covers.

In [7]:
q2_curve = st.coverage_curve(cost, weights, q2_picks, THRESHOLD_MIN)
target = st.min_k_for_target(q2_curve, TARGET_RELATIVE, relative_to_ceiling=True)
K_STAR = target["k"]
in_range = q2_curve["k"].between(*ELBOW_RANGE)
ELBOW_K = st.find_elbow(q2_curve[in_range])

if K_STAR is None:
    q2_stations, q2_total_trucks, q2_by_region = [], None, None
else:
    star_sites = q2_picks[:K_STAR]
    star = score(star_sites)
    star_trucks, star_peaks, star_fires = trucks_and_peaks(star, star_sites)
    q2_stations = [{"cand_id": int(site), "k": k, "trucks": int(star_trucks[site]),
                    "fires_covered": int(star_fires[site]), "peak_active": int(star_peaks[site])}
                   for k, site in enumerate(star_sites, start=1)]
    q2_total_trucks = int(star_trucks.sum())
    star_regions = pd.Series(regions_cand[star_sites], index=star_sites)
    q2_by_region = {region: {**region_scores(star["by_region"])[region],
                             "stations": int((star_regions == region).sum()),
                             "trucks": int(star_trucks[star_regions.index[star_regions == region]].sum())}
                    for region in REGIONS}

files["q2_coverage.json"] = {
    **header(), "fitted_on": files["meta.json"]["data"]["years"], "method": "greedy_coverage", "k_max": Q2_K_MAX,
    "target_relative": TARGET_RELATIVE, "k_star": K_STAR, "elbow_k": int(ELBOW_K), "elbow_range": ELBOW_RANGE,
    "picks": [int(s) for s in q2_picks],
    "curve": [{"k": int(r.k), "site": int(r.site), "coverage": share(r.coverage), "relative": share(r.coverage_relative),
               "mean_min": minutes(r.mean_minutes)} for r in q2_curve.itertuples(index=False)],
    "stations": q2_stations, "total_trucks": q2_total_trucks, "by_region_at_k_star": q2_by_region}
print(f"k_star = {K_STAR}, elbow_k = {ELBOW_K} (K {ELBOW_RANGE[0]}-{ELBOW_RANGE[1]}), total trucks = {q2_total_trucks}")
if K_STAR is not None:
    print(f"At k_star: coverage {star['coverage']:.4f}, relative {star['coverage_relative']:.4f}, mean {star['mean_minutes']:.1f} min; "
          f"trucks per station: max {star_trucks.max()}, "
          f"{ {n: int((star_trucks == n).sum()) for n in (1, 2, 3)} } and {(star_trucks >= 4).sum()} with 4+")
    display(pd.DataFrame(q2_by_region).T)

k_star = 81, elbow_k = 23 (K 1-60), total trucks = 214
At k_star: coverage 0.6484, relative 0.9510, mean 79.1 min; trucks per station: max 8, {1: 5, 2: 37, 3: 27} and 12 with 4+


,coverage,relative,stations,trucks
Cariboo,0.6913,0.9396,9.0,24.0
Coastal,0.8175,0.9423,13.0,40.0
Kamloops,0.9194,0.9597,15.0,41.0
Northwest,0.6532,0.9534,10.0,28.0
Prince George,0.4080,0.9628,20.0,46.0
Southeast,0.8203,0.9358,14.0,35.0


## 7. `q3_halls.json`
The 521 existing halls: score, truck needs, the 2× allocation, the optimized network that matches them,
and the next 20 stations to add.

In [8]:
halls_score = score(halls)
hall_trucks, hall_peaks, hall_fires = trucks_and_peaks(halls_score, halls)
hall_load = np.where(hall_fires.to_numpy() > 0, hall_trucks.to_numpy(), 0)   # contract: 0 for halls with no covered fire
BUDGET_2X = HALL_BUDGET_FACTOR * len(halls)
hall_alloc = st.allocate_trucks(hall_load, BUDGET_2X)

reaching = np.flatnonzero(q2_curve["coverage"].to_numpy() >= halls_score["coverage"])
if len(reaching) == 0:
    matching = None
else:
    K_MATCH = int(q2_curve["k"].iloc[reaching[0]])
    match = score(q2_picks[:K_MATCH])
    matching = {"k": K_MATCH, "coverage": share(match["coverage"]), "relative": share(match["coverage_relative"]),
                "mean_min": minutes(match["mean_minutes"]),
                "note": "first K of q2_coverage.picks whose coverage >= halls' coverage"}

next_stations, before = [], halls_score["coverage"]
for rank, site in enumerate(next_picks, start=1):
    after = score(np.concatenate([halls, next_picks[:rank]]))["coverage"]
    next_stations.append({"rank": rank, "cand_id": int(site), "coverage_after": share(after), "gain": share(after - before)})
    before = after

hall_counts = pd.Series(regions_cand[halls]).value_counts()
files["q3_halls.json"] = {
    **header(), "fitted_on": files["meta.json"]["data"]["years"],
    "score": {"coverage": share(halls_score["coverage"]), "relative": share(halls_score["coverage_relative"]),
              "mean_min": minutes(halls_score["mean_minutes"]),
              "by_region": {region: {**region_scores(halls_score["by_region"])[region], "halls": int(hall_counts.get(region, 0))}
                            for region in REGIONS}},
    "trucks": {"need_total": int(hall_trucks.sum()), "budget_2x": int(BUDGET_2X)},
    "halls": [{"cand_id": int(site), "fires_covered": int(hall_fires[site]), "trucks_need": int(hall_trucks[site]),
               "trucks_2x": int(allocated)} for site, allocated in zip(halls, hall_alloc)],
    "matching": matching, "next_stations": next_stations}
print(f"Halls: coverage {halls_score['coverage']:.4f}, relative {halls_score['coverage_relative']:.4f}, "
      f"mean {halls_score['mean_minutes']:.1f} min; trucks needed {hall_trucks.sum()}, 2x budget {BUDGET_2X}; "
      f"{(hall_fires == 0).sum()} halls cover no fire")
print(f"Matching: {matching}")
display(pd.DataFrame(next_stations).head(5).assign(name=lambda t: candidates.loc[t["cand_id"], "name"].to_numpy(),
                                                   region=lambda t: regions_cand[t["cand_id"]]))

Halls: coverage 0.5702, relative 0.8363, mean 83.4 min; trucks needed 663, 2x budget 1042; 192 halls cover no fire
Matching: {'k': 56, 'coverage': 0.5724, 'relative': 0.8396, 'mean_min': 89.9, 'note': "first K of q2_coverage.picks whose coverage >= halls' coverage"}


,rank,cand_id,coverage_after,gain,name,region
0,1,843,0.5840,0.0138,Wonowon,Prince George
1,2,815,0.5959,0.0119,Middle River,Prince George
2,3,769,0.6047,0.0087,Anahim Lake,Cariboo
3,4,827,0.6127,0.0080,Babine,Northwest
4,5,822,0.6198,0.0071,Tumbler Ridge,Prince George


## 8. `gaps.json`

In [9]:
gap_rows = []
for region in REGIONS:
    inside = regions_demand == region
    gap_rows.append({"region": region, "fires": int(inside.sum()), "unreachable_fires": int((inside & ~reachable).sum()),
                     "unreachable_weight_share": share(1 - CEILING_BY_REGION[region])})
files["gaps.json"] = {
    **header(), "threshold_min": THRESHOLD_MIN,
    "overall": {"fires": len(demand), "unreachable_fires": int((~reachable).sum()),
                "unreachable_weight_share": share(1 - CEILING)},
    "by_region": gap_rows,
    "thresholds": [{"threshold_min": int(row["threshold_min"]),
                    "reach_km": round((row["threshold_min"] - config["DISPATCH_MIN"]) / 60 * config["SPEED_KMH"] / config["DETOUR"], 2),
                    "ceiling": share(st.ceiling(cost, weights, row["threshold_min"]))} for row in results["thresholds"]]}
print(files["gaps.json"]["overall"], files["gaps.json"]["thresholds"])
display(pd.DataFrame(gap_rows).set_index("region"))

{'fires': 7032, 'unreachable_fires': 1935, 'unreachable_weight_share': 0.3182} [{'threshold_min': 30, 'reach_km': 16.67, 'ceiling': 0.2838}, {'threshold_min': 60, 'reach_km': 41.67, 'ceiling': 0.6818}, {'threshold_min': 90, 'reach_km': 66.67, 'ceiling': 0.8452}]


,fires,unreachable_fires,unreachable_weight_share
region,,,
Cariboo,502,124,0.2642
Coastal,1000,88,0.1324
Kamloops,1020,42,0.0420
Northwest,927,231,0.3148
Prince George,2596,1304,0.5762
Southeast,987,146,0.1234


## 9. `evidence.json`
Copied from notebook 05's saved results with the renames and conversions in the contract's mapping table.
Nothing is recomputed here.

In [10]:
def csv(name):
    return pd.read_csv(RESULTS_DIR / name)


def or_none(value, convert):
    return None if pd.isna(value) else convert(value)


VARIANT_NAMES = {"capped": "capped_pmedian", "uncapped": "uncapped_pmedian"}
LAYOUT_NAMES = {"capped p-median": "capped_pmedian", "greedy coverage": "greedy_coverage",
                "most populous towns": "most_populous_towns", "random (median of 1000)": "random_median",
                "random (best of 1000)": "random_best"}
METHOD_NAMES = {"capped p-median": "capped_pmedian", "fair": "fair"}

pmedian = results["exact_pmedian"]
statuses = {row["status"] for row in pmedian["saved_results"]}
if statuses == {"timed out"}:
    pmedian_status, pmedian_note = "timed_out", f"too large to solve exactly in {pmedian['limit_s']} s"
elif statuses == {"optimal"}:
    pmedian_status, pmedian_note = "solved", "solved to optimality"
else:
    raise ValueError(f"Mixed or missing exact p-median statuses: {statuses}")

stations_05 = csv("q2_stations_trucks.csv")
worst = stations_05.sort_values(["overloaded_days_2023", "cand_id"], ascending=[False, True]).iloc[0]
comparison = csv("q3_comparison.csv")
optimized_train = comparison.loc[comparison["layout"].str.startswith("optimized"), "train_coverage"]
assert len(optimized_train) == 1
stability = csv("loyo_region_stability.csv")
fold_columns = sorted(column for column in stability.columns if column.startswith("without_"))

files["evidence.json"] = {
    **header(),
    "split": {"train": config["TRAIN_YEARS"], "test": config["TEST_YEARS"][0]},
    "q1_variants": [{"variant": VARIANT_NAMES[row["variant"]], "k": int(row["k"]),
                     "train_coverage": share(row["train_coverage"]), "train_relative": share(row["train_relative"]),
                     "test_coverage": share(row["test_coverage"]), "test_relative": share(row["test_relative"])}
                    for row in results["q1"]],
    "baselines": [{"k": int(r.k), "layout": LAYOUT_NAMES[r.layout], "test_relative": share(r.test_coverage_relative),
                   "test_mean_min": minutes(r.test_mean_minutes),
                   "beats_random_coverage_share": or_none(r.beats_random_pct_coverage, lambda v: share(v / 100)),
                   "beats_random_minutes_share": or_none(r.beats_random_pct_minutes, lambda v: share(v / 100))}
                  for r in csv("baselines.csv").itertuples(index=False)],
    "loyo": {
        "k": int(results["loyo"]["k"]),
        "folds": [{"held_out_year": int(r.held_out_year), "coverage": share(r.held_out_coverage),
                   "ceiling": share(r.held_out_ceiling), "relative": share(r.held_out_relative)}
                  for r in csv("loyo_results.csv").itertuples(index=False)],
        "mean_jaccard": round(float(results["loyo"]["mean_pairwise_jaccard"]), 4),
        "region_stability": [{"region": row["region"], "per_fold": [int(row[column]) for column in fold_columns],
                              "mean": round(float(row["mean"]), 2), "std": round(float(row["std"]), 2)}
                             for _, row in stability.iterrows()]},
    "exact_check": {
        "coverage": [{"k": int(r.k), "greedy": share(r.greedy_coverage), "exact": share(r.exact_coverage),
                      "gap_points": round(float(r.gap_points), 4)} for r in csv("exact_coverage_check.csv").itertuples(index=False)],
        "pmedian": {"status": pmedian_status, "variables": int(pmedian["variables"]), "note": pmedian_note}},
    "fairness": [{"k": int(r.k), "method": METHOD_NAMES[r.method], "coverage": share(r.coverage),
                  "lowest_region": r.lowest_region, "lowest_relative": share(r.lowest_region_relative)}
                 for r in csv("fairness.csv").itertuples(index=False)],
    "weight_sensitivity": {"k": int(results["weight_sensitivity"]["k"]),
                           "jaccard": round(float(results["weight_sensitivity"]["jaccard"]), 4),
                           "coverage_weighted": share(results["weight_sensitivity"]["coverage_weighted_picks"]),
                           "coverage_unweighted": share(results["weight_sensitivity"]["coverage_unweighted_picks"])},
    "overload_2023": {
        "k_star_train": int(results["q2"]["k_star"]), "total_trucks_train": int(results["q2"]["total_trucks"]),
        "overloaded_share": share(results["q2"]["overload_share_2023"]),
        "stations_ever_overloaded": int(results["q2"]["stations_ever_overloaded_2023"]),
        "worst_day_overloaded": int(results["q2"]["worst_day_overloaded_2023"]),
        "worst_station": {"cand_id": int(worst["cand_id"]), "trucks": int(worst["trucks"]),
                          "peak_active": int(worst["peak_active_2023"]), "overloaded_days": int(worst["overloaded_days_2023"])}},
    "q3_split": {"halls_train": share(results["q3"]["halls_train_coverage"]), "halls_test": share(results["q3"]["halls_test_coverage"]),
                 "k_match_train": int(results["q3"]["k_matching_halls"]), "optimized_train": share(optimized_train.iloc[0]),
                 "optimized_test": share(results["q3"]["matching_test_coverage"])},
}
evidence = files["evidence.json"]
print({key: (len(value) if isinstance(value, list) else "object") for key, value in evidence.items() if key not in header()})
print(evidence["overload_2023"], evidence["q3_split"], evidence["exact_check"]["pmedian"])

{'split': 'object', 'q1_variants': 8, 'baselines': 15, 'loyo': 'object', 'exact_check': 'object', 'fairness': 6, 'weight_sensitivity': 'object', 'overload_2023': 'object', 'q3_split': 'object'}
{'k_star_train': 78, 'total_trucks_train': 204, 'overloaded_share': 0.0121, 'stations_ever_overloaded': 31, 'worst_day_overloaded': 6, 'worst_station': {'cand_id': 687, 'trucks': 2, 'peak_active': 11, 'overloaded_days': 29}} {'halls_train': 0.6532, 'halls_test': 0.4238, 'k_match_train': 53, 'optimized_train': 0.6546, 'optimized_test': 0.3633} {'status': 'timed_out', 'variables': 956524, 'note': 'too large to solve exactly in 120 s'}


## 10. Checks (contract section 4)
Everything is serialized first, compact and with `allow_nan=False`. The conformance check re-reads the
serialized `fires.json` and `candidates.json` and re-scores the K = 20 layouts with an independent
implementation of the contract's formula. If any check fails, nothing is written.

In [11]:
serialized = {name: json.dumps(payload, separators=(",", ":"), allow_nan=False, ensure_ascii=False).encode("utf-8")
              for name, payload in files.items()}   # check 7: allow_nan=False raises on NaN or Infinity
parsed = {name: json.loads(text) for name, text in serialized.items()}
q1, q2, q3 = parsed["q1_layouts.json"], parsed["q2_coverage.json"], parsed["q3_halls.json"]
cand_list, fire_list = parsed["candidates.json"]["candidates"], parsed["fires.json"]["fires"]
valid_ids = {c["cand_id"] for c in cand_list}

pick_lists = {"q1 capped_pmedian": q1["variants"]["capped_pmedian"]["picks"], "q1 fair": q1["variants"]["fair"]["picks"],
              "q2": q2["picks"], "q3 next_stations": [s["cand_id"] for s in q3["next_stations"]]}
referenced = set().union(*pick_lists.values(), q1["variants"]["fair"]["seeds"], [s["cand_id"] for s in q2["stations"]],
                         [h["cand_id"] for h in q3["halls"]], [parsed["evidence.json"]["overload_2023"]["worst_station"]["cand_id"]],
                         *[[e["site"] for e in v["curve"]] for v in q1["variants"].values()], [e["site"] for e in q2["curve"]])


def contract_score(open_ids):
    """Independent re-implementation of contract section 2, from the serialized files only."""
    settings = parsed["meta.json"]["settings"]
    total = covered = weighted = 0.0
    sites = [(np.radians(cand_list[i]["lat"]), np.radians(cand_list[i]["lon"])) for i in open_ids]
    for fire in fire_list:
        lat, lon = np.radians(fire["lat"]), np.radians(fire["lon"])
        best = min(2 * settings["earth_radius_km"]
                   * np.arcsin(np.sqrt(np.sin((s_lat - lat) / 2) ** 2 + np.cos(lat) * np.cos(s_lat) * np.sin((s_lon - lon) / 2) ** 2))
                   * settings["detour"] / settings["speed_kmh"] * 60 + settings["dispatch_min"] for s_lat, s_lon in sites)
        total += fire["weight"]
        weighted += fire["weight"] * best
        covered += fire["weight"] * (best <= settings["threshold_min"])
    coverage = covered / total
    return {"coverage": coverage, "relative": coverage / parsed["meta.json"]["ceiling"]["all_years"], "mean_min": weighted / total}


conformance = {}
for name, variant in q1["variants"].items():
    stored = next(e for e in variant["curve"] if e["k"] == CONFORMANCE_K)
    again = contract_score(variant["picks"][:CONFORMANCE_K])
    conformance[name] = {"coverage_diff": abs(again["coverage"] - stored["coverage"]),
                         "relative_diff": abs(again["relative"] - stored["relative"]),
                         "mean_min_diff": abs(again["mean_min"] - stored["mean_min"])}
display(pd.DataFrame(conformance).T)

static_ids = {f["fire_id"] for f in parsed["static_fires.json"]["fires"]}
other_bytes = sum(len(text) for name, text in serialized.items() if name != "fires.json")
fair_ks = [e["k"] for e in q1["variants"]["fair"]["curve"]]
checks = {
    "1. every cand_id referenced exists in candidates.json": referenced <= valid_ids,
    "1. every pick list is unique": all(len(p) == len(set(p)) for p in pick_lists.values()),
    "1. pick list lengths (60, 60, 200, 20)": [len(p) for p in pick_lists.values()] == [Q1_K_MAX, Q1_K_MAX, Q2_K_MAX, NEXT_N],
    "2. candidates sorted with cand_id == index, 851 entries": [c["cand_id"] for c in cand_list] == list(range(EXPECTED_CANDIDATES)),
    "2. halls are cand_id 0-520": [c["cand_id"] for c in cand_list if c["kind"] == "hall"] == list(range(EXPECTED_HALLS)),
    "3. fires.json has 7,032 entries": len(fire_list) == EXPECTED_FIRES,
    "3. static_fires.json has 195 entries, none in fires.json": len(static_ids) == EXPECTED_STATIC
        and not static_ids & {f["fire_id"] for f in fire_list},
    "3. reachable == (nearest_min <= threshold_min), before rounding": bool((reachable == (nearest <= THRESHOLD_MIN)).all())
        and [f["reachable"] for f in fire_list] == fire_rows["reachable"].tolist(),
    "3. fires sorted by date, then fire_id": [(f["date"], f["fire_id"]) for f in fire_list] == sorted((f["date"], f["fire_id"]) for f in fire_list),
    "4. q1 capped curve k = 1..60": [e["k"] for e in q1["variants"]["capped_pmedian"]["curve"]] == list(range(1, Q1_K_MAX + 1)),
    "4. q1 fair curve k = 6..60 (55 entries), seeds == picks[:6]": fair_ks == list(range(6, Q1_K_MAX + 1)) and len(fair_ks) == 55
        and q1["variants"]["fair"]["seeds"] == q1["variants"]["fair"]["picks"][:6],
    "4. q2 curve k = 1..200": [e["k"] for e in q2["curve"]] == list(range(1, Q2_K_MAX + 1)),
    "4. q2 coverage is non-decreasing": bool((np.diff(q2_curve["coverage"]) >= 0).all())
        and all(a["coverage"] <= b["coverage"] for a, b in zip(q2["curve"], q2["curve"][1:])),
    "4. curve sites follow the pick order": all(e["site"] == v["picks"][e["k"] - 1] for v in q1["variants"].values() for e in v["curve"])
        and all(e["site"] == q2["picks"][e["k"] - 1] for e in q2["curve"]),
    "5. q2.total_trucks == sum(stations.trucks)": q2["total_trucks"] == (sum(s["trucks"] for s in q2["stations"]) if q2["k_star"] else None),
    "5. q2.stations is the k_star layout in pick order": [s["cand_id"] for s in q2["stations"]] == q2["picks"][: q2["k_star"] or 0],
    "5. q3.trucks_2x sums to budget_2x": sum(h["trucks_2x"] for h in q3["halls"]) == q3["trucks"]["budget_2x"],
    "5. q3.need_total == sum(trucks_need); 521 halls": sum(h["trucks_need"] for h in q3["halls"]) == q3["trucks"]["need_total"]
        and [h["cand_id"] for h in q3["halls"]] == list(range(EXPECTED_HALLS)),
    f"6. conformance at K = {CONFORMANCE_K} (shares within {SHARE_TOLERANCE}, minutes within {MINUTES_TOLERANCE})":
        all(d["coverage_diff"] <= SHARE_TOLERANCE and d["relative_diff"] <= SHARE_TOLERANCE and d["mean_min_diff"] <= MINUTES_TOLERANCE
            for d in conformance.values()),
    "7. no NaN or Infinity (serialized with allow_nan=False)": True,
    "8. all files except fires.json total under 1 MB": other_bytes < MAX_OTHER_FILES_BYTES,
    "8. fires.json under 1.5 MB": len(serialized["fires.json"]) < MAX_FIRES_BYTES,
    "regions are the six contract names everywhere": {c["region"] for c in cand_list} | {f["region"] for f in fire_list} == set(REGIONS),
    "every file has schema_version 1 and generated_at": all(p["schema_version"] == SCHEMA_VERSION and p["generated_at"] == GENERATED_AT
                                                           for p in parsed.values()),
}
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")
ALL_CHECKS_PASS = all(checks.values())

,coverage_diff,relative_diff,mean_min_diff
capped_pmedian,0.00002,0.00005,0.013670
fair,0.00001,0.00004,0.040684


PASS  1. every cand_id referenced exists in candidates.json
PASS  1. every pick list is unique
PASS  1. pick list lengths (60, 60, 200, 20)
PASS  2. candidates sorted with cand_id == index, 851 entries
PASS  2. halls are cand_id 0-520
PASS  3. fires.json has 7,032 entries
PASS  3. static_fires.json has 195 entries, none in fires.json
PASS  3. reachable == (nearest_min <= threshold_min), before rounding
PASS  3. fires sorted by date, then fire_id
PASS  4. q1 capped curve k = 1..60
PASS  4. q1 fair curve k = 6..60 (55 entries), seeds == picks[:6]
PASS  4. q2 curve k = 1..200
PASS  4. q2 coverage is non-decreasing
PASS  4. curve sites follow the pick order
PASS  5. q2.total_trucks == sum(stations.trucks)
PASS  5. q2.stations is the k_star layout in pick order
PASS  5. q3.trucks_2x sums to budget_2x
PASS  5. q3.need_total == sum(trucks_need); 521 halls
PASS  6. conformance at K = 20 (shares within 0.0001, minutes within 0.1)
PASS  7. no NaN or Infinity (serialized with allow_nan=False)
PAS

## 11. Write
Only if every check passed.

In [12]:
if not ALL_CHECKS_PASS:
    raise RuntimeError("A contract check failed; nothing was written.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, text in serialized.items():
    (OUTPUT_DIR / name).write_bytes(text)
sizes = pd.Series({name: len(text) for name, text in serialized.items()}, name="bytes").sort_values(ascending=False)
display(sizes.to_frame().assign(kB=lambda t: (t["bytes"] / 1000).round(1)))
print(f"Wrote {len(serialized)} files to {OUTPUT_DIR.relative_to(REPO_ROOT)}; "
      f"fires.json {len(serialized['fires.json']) / 1e6:.2f} MB, all others {other_bytes / 1e6:.2f} MB; generated_at {GENERATED_AT}")

,bytes,kB
fires.json,1225332,1225.3
candidates.json,109735,109.7
q1_layouts.json,44867,44.9
q3_halls.json,35383,35.4
q2_coverage.json,21293,21.3
static_fires.json,13475,13.5
evidence.json,5694,5.7
meta.json,960,1.0
gaps.json,913,0.9


Wrote 9 files to web\public\data; fires.json 1.23 MB, all others 0.23 MB; generated_at 2026-10-04T13:58:55Z
